# Session 1: Fundamentals of Networking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/04-configuring-network-infrastructure/code/session-01-fundamentals-of-networking.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-01_delay_calc.py`

In [1]:
"""Session 1 - Measure what makes a network "good": delay, throughput and availability.

A file transfer is slowed by four delays: transmission (pushing bits onto the
wire), propagation (the signal travelling), processing and queuing.
This script calculates them for real link types so you can compare designs.
Run:  python session-01_delay_calc.py
"""

SPEED_OF_LIGHT_FIBRE = 2e8      # metres per second (about 2/3 of c in glass or copper)


def transmission_delay(size_bytes, bandwidth_bps):
    """Seconds needed to put every bit of the message onto the link."""
    return size_bytes * 8 / bandwidth_bps


def propagation_delay(distance_m, speed=SPEED_OF_LIGHT_FIBRE):
    """Seconds the first bit needs to travel the distance."""
    return distance_m / speed


def total_delay(size_bytes, bandwidth_bps, distance_m, processing_s=0.0001, queuing_s=0.0):
    t = transmission_delay(size_bytes, bandwidth_bps)
    p = propagation_delay(distance_m)
    return t, p, t + p + processing_s + queuing_s


def availability(mtbf_h, mttr_h):
    """Fraction of time a component works: MTBF / (MTBF + MTTR)."""
    return mtbf_h / (mtbf_h + mttr_h)


def downtime_per_year(avail):
    return (1 - avail) * 365 * 24 * 60   # minutes


if __name__ == "__main__":
    FILE = 50 * 1024 * 1024   # a 50 MB lecture video
    links = [
        ("Office LAN, Fast Ethernet, 50 m", 100e6, 50),
        ("Office LAN, Gigabit, 50 m", 1e9, 50),
        ("Campus fibre, 10 Gbps, 2 km", 10e9, 2_000),
        ("Karachi -> Lahore WAN, 100 Mbps, 1 200 km", 100e6, 1_200_000),
        ("Karachi -> London WAN, 100 Mbps, 6 300 km", 100e6, 6_300_000),
    ]
    print("Sending a 50 MB file over different links")
    print(f"{'Link':45} {'Transmit':>10} {'Propagate':>11} {'Total':>9}")
    for name, bw, dist in links:
        t, p, tot = total_delay(FILE, bw, dist)
        print(f"{name:45} {t:9.3f}s {p*1000:9.3f}ms {tot:8.3f}s")

    print("\nA small 1 500-byte packet (one Ethernet frame) on the same links")
    for name, bw, dist in links:
        t, p, tot = total_delay(1500, bw, dist)
        print(f"{name:45} transmit {t*1e6:8.1f} us   propagate {p*1000:7.3f} ms")

    print("\nReliability: availability and yearly downtime")
    for name, mtbf, mttr in [("Single access switch", 50_000, 8),
                             ("Consumer router", 10_000, 24),
                             ("Core switch with 4 h support", 200_000, 4)]:
        a = availability(mtbf, mttr)
        print(f"{name:32} A = {a*100:.4f}%   downtime = {downtime_per_year(a):7.1f} min/year")
    # two switches in parallel: down only if both are down
    a = availability(50_000, 8)
    pair = 1 - (1 - a) ** 2
    print(f"{'Two access switches in parallel':32} A = {pair*100:.6f}%   downtime = {downtime_per_year(pair)*60:7.3f} s/year")

Sending a 50 MB file over different links
Link                                            Transmit   Propagate     Total
Office LAN, Fast Ethernet, 50 m                   4.194s     0.000ms    4.194s
Office LAN, Gigabit, 50 m                         0.419s     0.000ms    0.420s
Campus fibre, 10 Gbps, 2 km                       0.042s     0.010ms    0.042s
Karachi -> Lahore WAN, 100 Mbps, 1 200 km         4.194s     6.000ms    4.200s
Karachi -> London WAN, 100 Mbps, 6 300 km         4.194s    31.500ms    4.226s

A small 1 500-byte packet (one Ethernet frame) on the same links
Office LAN, Fast Ethernet, 50 m               transmit    120.0 us   propagate   0.000 ms
Office LAN, Gigabit, 50 m                     transmit     12.0 us   propagate   0.000 ms
Campus fibre, 10 Gbps, 2 km                   transmit      1.2 us   propagate   0.010 ms
Karachi -> Lahore WAN, 100 Mbps, 1 200 km     transmit    120.0 us   propagate   6.000 ms
Karachi -> London WAN, 100 Mbps, 6 300 km     transmit    

## `session-01_device_roles.py`

In [2]:
"""Session 1 - Pick the right device for each networking job.

Each device works at a different layer and makes decisions with a different
address. Given a list of requirements, this script recommends a device and
explains why, then checks a small office design against the criteria of a
good network (performance, reliability, scalability, security, manageability).
Run:  python session-01_device_roles.py
"""

DEVICES = {
    "hub":          {"layer": 1, "uses": "nothing (repeats bits to every port)"},
    "switch":       {"layer": 2, "uses": "MAC addresses (MAC address table)"},
    "router":       {"layer": 3, "uses": "IP addresses (routing table)"},
    "multilayer switch": {"layer": 3, "uses": "MAC and IP addresses in hardware"},
    "firewall":     {"layer": 4, "uses": "IPs, ports and connection state (rules)"},
    "access point": {"layer": 2, "uses": "MAC addresses (bridges Wi-Fi to Ethernet)"},
    "wireless LAN controller": {"layer": 2, "uses": "AP inventory and policies (manages many APs centrally)"},
}

NEEDS = [
    ("Connect 24 PCs in one lab so they can share files", "switch"),
    ("Join the lab network to the internet", "router"),
    ("Route between 10 VLANs at wire speed in the server room", "multilayer switch"),
    ("Allow only HTTPS from the internet to the web server", "firewall"),
    ("Give laptops Wi-Fi in the library", "access point"),
    ("Manage 60 access points from one dashboard", "wireless LAN controller"),
]


def score_design(design):
    """Return (criterion, ok, reason) tuples for a simple design dictionary."""
    checks = [
        ("Performance", design["uplink_gbps"] >= 1, f"uplink {design['uplink_gbps']} Gbps"),
        ("Reliability", design["redundant_uplinks"] >= 2, f"{design['redundant_uplinks']} uplink(s)"),
        ("Scalability", design["free_ports_pct"] >= 20, f"{design['free_ports_pct']}% ports free"),
        ("Security", design["guest_isolated"] and not design["telnet_enabled"],
         f"guest isolated={design['guest_isolated']}, telnet={design['telnet_enabled']}"),
        ("Manageability", design["central_logging"], f"central logging={design['central_logging']}"),
    ]
    return checks


if __name__ == "__main__":
    print("Requirement -> recommended device")
    for need, dev in NEEDS:
        d = DEVICES[dev]
        print(f" - {need}\n     => {dev} (layer {d['layer']}, decides using {d['uses']})")

    for name, design in [
        ("Version 1 (first draft)", dict(uplink_gbps=0.1, redundant_uplinks=1, free_ports_pct=5,
                                         guest_isolated=False, telnet_enabled=True, central_logging=False)),
        ("Version 2 (improved)", dict(uplink_gbps=10, redundant_uplinks=2, free_ports_pct=30,
                                      guest_isolated=True, telnet_enabled=False, central_logging=True)),
    ]:
        checks = score_design(design)
        passed = sum(ok for _, ok, _ in checks)
        print(f"\n{name}: {passed}/{len(checks)} criteria met")
        for crit, ok, why in checks:
            print(f"   [{'PASS' if ok else 'FAIL'}] {crit:13} {why}")

Requirement -> recommended device
 - Connect 24 PCs in one lab so they can share files
     => switch (layer 2, decides using MAC addresses (MAC address table))
 - Join the lab network to the internet
     => router (layer 3, decides using IP addresses (routing table))
 - Route between 10 VLANs at wire speed in the server room
     => multilayer switch (layer 3, decides using MAC and IP addresses in hardware)
 - Allow only HTTPS from the internet to the web server
     => firewall (layer 4, decides using IPs, ports and connection state (rules))
 - Give laptops Wi-Fi in the library
     => access point (layer 2, decides using MAC addresses (bridges Wi-Fi to Ethernet))
 - Manage 60 access points from one dashboard
     => wireless LAN controller (layer 2, decides using AP inventory and policies (manages many APs centrally))

Version 1 (first draft): 0/5 criteria met
   [FAIL] Performance   uplink 0.1 Gbps
   [FAIL] Reliability   1 uplink(s)
   [FAIL] Scalability   5% ports free
   [FAIL]